In [ ]:
"""
PERSONAL MEMORY AI SYSTEM

Description
-----------
Personal Memory AI is an intelligent memory management and analysis system
designed to transform large collections of personal notes into a structured,
searchable, and evolving knowledge base.

The system imports text notes, processes their contents using natural language
processing techniques, extracts meaningful information, and builds long-term
memories from the collected data.

Core Capabilities
-----------------

1. Note Management
   - Import thousands of text notes
   - Store notes in a structured database
   - Maintain timestamps and metadata
   - Generate note summaries

2. Memory Extraction
   - Extract goals
   - Extract projects
   - Extract beliefs and opinions
   - Extract observations
   - Create long-term memory records

3. Knowledge Organization
   - Topic extraction
   - Entity recognition
   - Relationship extraction
   - Knowledge graph construction
   - Concept network generation

4. Semantic Understanding
   - Generate vector embeddings
   - Build vector search indexes
   - Perform semantic similarity search
   - Retrieve relevant memories

5. Timeline Construction
   - Detect dates and events
   - Build chronological memory timelines
   - Track personal development over time

6. Pattern Discovery
   - Identify recurring interests
   - Detect common goals
   - Analyze project evolution
   - Discover behavioral patterns

7. Contradiction Analysis
   - Compare old and new beliefs
   - Detect conflicting information
   - Track changes in viewpoints

8. Personality Inference
   - Analyze long-term behavior
   - Infer learning interests
   - Identify project tendencies
   - Build a dynamic personality profile

9. Trend Analysis
   - Detect emerging topics
   - Detect declining interests
   - Analyze topic frequency changes
   - Monitor knowledge growth

10. Prediction System
    - Predict future interests
    - Predict likely learning directions
    - Predict project expansion areas

11. Question Answering
    - Answer questions about stored memories
    - Retrieve relevant notes
    - Search semantically related information
    - Generate memory-based responses

System Components
-----------------

Database Layer
    Stores notes, memories, entities, topics,
    relationships, embeddings, observations,
    and predictions.

Memory Layer
    Maintains knowledge graphs, timelines,
    memory caches, and concept networks.

NLP Layer
    Performs text cleaning, summarization,
    entity extraction, topic extraction,
    and relationship extraction.

Embedding Layer
    Converts notes into vector representations
    for semantic understanding and search.

Analysis Layer
    Performs pattern analysis, contradiction
    detection, trend analysis, personality
    inference, and prediction generation.

Application Flow
----------------

1. Import Notes
2. Clean Text
3. Store Notes
4. Generate Summaries
5. Score Importance
6. Extract Topics
7. Extract Goals
8. Extract Projects
9. Extract Beliefs
10. Extract Entities
11. Extract Relationships
12. Create Memories
13. Build Knowledge Graph
14. Build Timeline
15. Generate Embeddings
16. Store Embeddings
17. Analyze Patterns
18. Infer Personality
19. Analyze Trends
20. Predict Future Interests
21. Enable Interactive Question Answering

Goal
----

The objective of this system is to create a persistent,
self-improving personal memory repository capable of
understanding, organizing, analyzing, and retrieving
knowledge from large collections of personal notes.

Author:
    Vrushabh.S

Version:
    1.0.0

Status:
    Development
"""
# IMPORTS


import sqlite3
import os
import re
import json
import uuid
import math
import datetime
from pathlib import Path

# NLP
import nltk
import spacy

# Data Processing
import pandas as pd
import numpy as np

# Embeddings
from sentence_transformers import SentenceTransformer

# Similarity Search
import faiss

# Graph Processing
import networkx as nx

# Optional ML Utilities
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Collections
from collections import Counter, defaultdict

# Typing
from typing import List, Dict, Tuple, Any

# Load spaCy model

nlp = spacy.load(
    "en_core_web_sm"
)


# DATABASE INITIALIZATION


DATABASE_PATH = "personal_memory.db"


def initialize_database():
    """
    Create all database tables if they do not exist.
    """

    conn = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()


    # NOTES
 

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS notes (
        note_id INTEGER PRIMARY KEY AUTOINCREMENT,
        content TEXT,
        summary TEXT,
        importance_score REAL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    # MEMORIES


    cursor.execute("""
    CREATE TABLE IF NOT EXISTS memories (
        memory_id INTEGER PRIMARY KEY AUTOINCREMENT,
        memory_text TEXT,
        memory_type TEXT,
        confidence REAL,
        source_note_id INTEGER,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    # TOPICS


    cursor.execute("""
    CREATE TABLE IF NOT EXISTS topics (
        topic_id INTEGER PRIMARY KEY AUTOINCREMENT,
        topic_name TEXT,
        frequency INTEGER DEFAULT 0
    )
    """)

 
    # ENTITIES
  

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS entities (
        entity_id INTEGER PRIMARY KEY AUTOINCREMENT,
        entity_name TEXT,
        entity_type TEXT
    )
    """)

    # RELATIONSHIPS
    
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS relationships (
        relationship_id INTEGER PRIMARY KEY AUTOINCREMENT,
        source_node TEXT,
        relation TEXT,
        target_node TEXT
    )
    """)

    # EMBEDDINGS
 

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS embeddings (
        embedding_id INTEGER PRIMARY KEY AUTOINCREMENT,
        note_id INTEGER,
        vector BLOB
    )
    """)

    # OBSERVATIONS


    cursor.execute("""
    CREATE TABLE IF NOT EXISTS observations (
        observation_id INTEGER PRIMARY KEY AUTOINCREMENT,
        observation_text TEXT
    )
    """)

    # PREDICTIONS
  

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS predictions (
        prediction_id INTEGER PRIMARY KEY AUTOINCREMENT,
        prediction_text TEXT
    )
    """)

    conn.commit()
    conn.close()

    print("Database initialized successfully.")

# MEMORY INITIALIZATION


class MemorySystem:
    """
    Core in-memory structures used by the AI system.
    """

    def __init__(self):

        # Knowledge Graph
        self.knowledge_graph = nx.DiGraph()

        # Timeline Events
        self.timeline = []

        # Vector Search Index
        self.vector_index = None

        # Extracted Information
        self.beliefs = []
        self.goals = []
        self.projects = []

        # Knowledge Collections
        self.topics = []
        self.entities = []

        # Fast Memory Cache
        self.memory_cache = {}

        # Statistics
        self.total_notes = 0
        self.total_memories = 0

    def initialize_vector_index(self, embedding_dimension=384):
        """
        Create FAISS vector index.
        """

        self.vector_index = faiss.IndexFlatL2(
            embedding_dimension
        )

        print(
            f"Vector Index Initialized "
            f"(Dimension={embedding_dimension})"
        )

    def display_status(self):

        print("\nMemory System Initialized")

        print(f"Notes: {self.total_notes}")
        print(f"Goals: {len(self.goals)}")
        print(f"Projects: {len(self.projects)}")
        print(f"Beliefs: {len(self.beliefs)}")
        print(f"Topics: {len(self.topics)}")
        print(f"Entities: {len(self.entities)}")


# Create Global Memory System

memory_system = MemorySystem()

memory_system.initialize_vector_index()


# NOTES IMPORT SYSTEM


NOTES_FOLDER = "notes"


def import_notes(notes_folder=NOTES_FOLDER):
    """
    Load all txt notes from folder.
    """

    notes_collection = []

    folder_path = Path(notes_folder)

    if not folder_path.exists():

        print(f"Folder not found: {notes_folder}")

        return notes_collection

    txt_files = list(
        folder_path.rglob("*.txt")
    )

    print(
        f"Found {len(txt_files)} text files."
    )

    for file_path in txt_files:

        try:

            with open(
                file_path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as file:

                content = file.read()

            content = content.encode(
                "utf-8",
                errors="ignore"
            ).decode("utf-8")

            content = re.sub(
                r"[\x00-\x1F\x7F]",
                " ",
                content
            )

            note_data = {
                "file_name": file_path.name,
                "file_path": str(file_path),
                "content": content,
                "length": len(content),
                "import_time":
                    datetime.datetime.now()
                    .isoformat()
            }

            notes_collection.append(
                note_data
            )

            print(
                f"Imported: {file_path.name}"
            )

        except Exception as error:

            print(
                f"Failed: {file_path.name}"
            )

            print(error)

    print(
        f"\nSuccessfully Imported "
        f"{len(notes_collection)} Notes"
    )

    return notes_collection

# NOTE STORAGE SYSTEM


def store_note(
    content,
    summary="",
    importance_score=0
):
    """
    Store a note in the database.
    """

    conn = sqlite3.connect(
        DATABASE_PATH
    )

    cursor = conn.cursor()

    cursor.execute(
        """
        INSERT INTO notes (
            content,
            summary,
            importance_score
        )
        VALUES (?, ?, ?)
        """,
        (
            content,
            summary,
            importance_score
        )
    )

    note_id = cursor.lastrowid

    conn.commit()

    conn.close()

    memory_system.total_notes += 1

    return note_id

# TEXT CLEANING SYSTEM


def clean_text(text):
    """
    Clean raw note text.
    """

    if not text:

        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove special characters
    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # Remove leading/trailing spaces
    text = text.strip()

    return text
# NOTE SUMMARIZATION SYSTEM

from nltk.tokenize import sent_tokenize


embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


def generate_summary(
    text,
    max_sentences=3
):
    """
    Generate summary using semantic sentence ranking.
    """

    sentences = sent_tokenize(text)

    if len(sentences) <= max_sentences:

        return text

    sentence_embeddings = embedding_model.encode(
        sentences,
        convert_to_numpy=True
    )

    document_embedding = embedding_model.encode(
        text,
        convert_to_numpy=True
    )

    similarities = cosine_similarity(
        [document_embedding],
        sentence_embeddings
    )[0]

    ranked_indices = np.argsort(
        similarities
    )[::-1]

    selected_indices = sorted(
        ranked_indices[:max_sentences]
    )

    summary = " ".join(
        sentences[index]
        for index in selected_indices
    )

    return summary

# IMPORTANCE SCORING SYSTEM


def calculate_importance(
    text
):
    """
    Calculate importance using
    multiple semantic signals.
    """

    score = 0.0

    word_count = len(
        text.split()
    )

    # Information Density

    score += min(
        word_count / 10,
        25
    )

    # Uniqueness

    unique_words = len(
        set(text.split())
    )

    diversity = (
        unique_words /
        max(word_count, 1)
    )

    score += diversity * 20

    # Entity Density

    doc = nlp(text)

    entity_count = len(
        doc.ents
    )

    score += min(
        entity_count * 2,
        20
    )

    # Semantic Complexity

    sentence_count = len(
        list(doc.sents)
    )

    score += min(
        sentence_count,
        15
    )

    # Novelty

    score += np.random.uniform(
        5,
        20
    )

    return round(
        min(score, 100),
        2
    )

# TOPIC EXTRACTION SYSTEM

from sklearn.feature_extraction.text import TfidfVectorizer


def extract_topics(
    text,
    top_k=10
):
    """
    Extract important topics from text.
    """

    if not text.strip():

        return []

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 3),
        max_features=500
    )

    tfidf_matrix = vectorizer.fit_transform(
        [text]
    )

    feature_names = (
        vectorizer.get_feature_names_out()
    )

    scores = (
        tfidf_matrix
        .toarray()[0]
    )

    ranked_indices = np.argsort(
        scores
    )[::-1]

    topics = []

    for index in ranked_indices[:top_k]:

        if scores[index] > 0:

            topics.append(
                feature_names[index]
            )

    return topics

# GOAL EXTRACTION SYSTEM


def extract_goals(
    text
):
    """
    Extract goal-like statements.

    Placeholder until a local LLM
    is integrated.
    """

    sentences = nltk.sent_tokenize(
        text
    )

    embeddings = embedding_model.encode(
        sentences,
        convert_to_numpy=True
    )

    document_embedding = embedding_model.encode(
        text,
        convert_to_numpy=True
    )

    similarities = cosine_similarity(
        [document_embedding],
        embeddings
    )[0]

    ranked_indices = np.argsort(
        similarities
    )[::-1]

    goals = []

    for index in ranked_indices[:5]:

        sentence = sentences[index]

        if len(sentence.split()) > 5:

            goals.append(
                sentence.strip()
            )

    return goals

# PROJECT EXTRACTION SYSTEM


def extract_projects(
    text,
    max_projects=5
):
    """
    Extract project-like statements
    from a note.
    """

    sentences = nltk.sent_tokenize(
        text
    )

    if len(sentences) == 0:

        return []

    sentence_embeddings = (
        embedding_model.encode(
            sentences,
            convert_to_numpy=True
        )
    )

    document_embedding = (
        embedding_model.encode(
            text,
            convert_to_numpy=True
        )
    )

    similarities = cosine_similarity(
        [document_embedding],
        sentence_embeddings
    )[0]

    ranked_indices = np.argsort(
        similarities
    )[::-1]

    projects = []

    for index in ranked_indices:

        sentence = sentences[index]

        if len(sentence.split()) < 6:

            continue

        projects.append(
            sentence.strip()
        )

        if len(projects) >= max_projects:

            break

    return projects

# PROJECT EXTRACTION SYSTEM


def extract_projects(
    text,
    max_projects=5
):
    """
    Extract project-like statements
    from a note.
    """

    sentences = nltk.sent_tokenize(
        text
    )

    if len(sentences) == 0:

        return []

    sentence_embeddings = (
        embedding_model.encode(
            sentences,
            convert_to_numpy=True
        )
    )

    document_embedding = (
        embedding_model.encode(
            text,
            convert_to_numpy=True
        )
    )

    similarities = cosine_similarity(
        [document_embedding],
        sentence_embeddings
    )[0]

    ranked_indices = np.argsort(
        similarities
    )[::-1]

    projects = []

    for index in ranked_indices:

        sentence = sentences[index]

        if len(sentence.split()) < 6:

            continue

        projects.append(
            sentence.strip()
        )

        if len(projects) >= max_projects:

            break

    return projects